# server_test — the REBOUND web server in pure Rust

Starts the ported web server, fetches the /simulation binary over HTTP, then shuts it down with the 'Q' key endpoint. The served blob is a valid REBOUND binary that the C build loads to the identical state.

This notebook is self-contained: it builds the example with cargo, runs it, and shows the result. Everything it does can also be done by hand in a terminal:

```
cd rebound_rust
cargo build --release --example server_test
cd porttest
../target/release/examples/server_test
```

In [1]:
import os, subprocess
EXE = ".exe" if os.name == "nt" else ""   # platform executable suffix
NB_DIR  = os.getcwd()                       # <crate>/notebooks
ROOT    = os.path.dirname(os.path.dirname(NB_DIR))
CRATE   = os.path.join(ROOT, "rebound_rust")
WORK    = os.path.join(CRATE, "porttest")
if not os.path.exists(os.path.join(CRATE, "Cargo.toml")):
    raise SystemExit(
        "Could not find the crate. Run this notebook from "
        "the notebooks folder of a full checkout: " + CRATE)
# The example that is BUILT and RUN. It is usually the one the
# notebook is named after; where it differs (the stock
# shearing_sheet integrates forever by design) the terminating
# variant is used instead, and the note above says so.
EXAMPLE = "server_test"
OUTFILE = None
os.makedirs(WORK, exist_ok=True)
res = subprocess.run(["cargo", "build", "--release", "--example", EXAMPLE],
                     cwd=CRATE, capture_output=True, text=True)
print(res.stderr.strip()[-400:] or "build ok")

    Finished `release` profile [optimized] target(s) in 0.00s


In [2]:
import subprocess, time, urllib.request
exe = os.path.join(CRATE, "target", "release", "examples", "server_test" + EXE)
proc = subprocess.Popen([exe], cwd=WORK, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True)
time.sleep(2.0)
blob = urllib.request.urlopen("http://localhost:12873/simulation", timeout=10).read()
print(f"/simulation returned {len(blob)} bytes")
print(f"header: {blob[:32]!r}")
urllib.request.urlopen("http://localhost:12873/keyboard/81", timeout=10).read()
proc.wait(timeout=15)
print("server exited cleanly")


/simulation returned 3448 bytes
header: b'REBOUND Binary File. Version: 5.'
server exited cleanly
